<a href="https://colab.research.google.com/github/greeshmapj/AI-ML-nov2025-assignments/blob/main/Greeshma_Assignment6_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**1. Import Libraries**

In [137]:
import pandas as pd
import numpy as np

import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#**2. Load Dataset**

In [138]:
df = pd.read_csv('/content/tweets.csv')
df.head(3)

,id,label,tweet
0,1,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,2,0,Finally a transparant silicon case ^^ Thanks t...
2,3,0,We love this! Would you go? #talk #makememorie...


In [139]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7920 entries, 0 to 7919
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      7920 non-null   int64 
 1   label   7920 non-null   int64 
 2   tweet   7920 non-null   object
dtypes: int64(2), object(1)
memory usage: 185.8+ KB


**INSIGHT**
* id column present(not important)
* The tweet column is the only text feature and requires preprocessing such as cleaning and vectorization.
* The target variable(label) is binary (0 and 1), indicating a classification problem.
* No missing values present

#**3. Data Preprocessing**




##**3.1. Drop Unnecessary Column**

In [140]:
df.drop(['id'],axis=1,inplace = True)
df.head(3)

,label,tweet
0,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,0,Finally a transparant silicon case ^^ Thanks t...
2,0,We love this! Would you go? #talk #makememorie...


##**3.2. Remove Duplicates**

In [141]:
df.duplicated(subset=['tweet']).sum()

np.int64(2)

In [142]:
df = df.drop_duplicates(subset=['tweet'])
df.duplicated(subset=['tweet']).sum()

np.int64(0)

##**3.3. Check Labels**

In [143]:
print(df['label'].value_counts())

label
0    5892
1    2026
Name: count, dtype: int64


**INSIGHT**
* class 0 - 70%  (Positive)
* class 1 - 30%  (Negative)
* Indicates moderate imbalance(no SMOTE needed)


#**4. Text Cleaning**

In [144]:
#Remove stopwors
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

#text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)


#Apply fn
df['clean_text'] = df['tweet'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



#**5. Applying TF-IDF for Feature Engineering**

In [145]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

#**6. Train-Test Split**

In [146]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#**7. Train Model**

In [147]:
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

#**8. Predictions**

In [148]:
y_pred = model.predict(X_test)

#**9. Evaluation**

In [149]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8642676767676768


**INSIGHT**
* Model Accuracy = 86.42%
* Implies good baseline
* But may predict majority class more often due to class imbalance





In [150]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91      1179
           1       0.81      0.61      0.70       405

    accuracy                           0.86      1584
   macro avg       0.85      0.78      0.80      1584
weighted avg       0.86      0.86      0.86      1584



**INSIGHT**
* For class 0 - High recall(0.95) - Model is excellent at detecting class 0
* For class 1 - low recall - Model is missing many actual class 1 tweets

In [151]:
print(confusion_matrix(y_test, y_pred))

[[1123   56]
 [ 159  246]]


**INSIGHT**
* 1123 → Correctly predicted class 0 (True Negatives)
* 246 → Correctly predicted class 1 (True Positives)
* 56 → False Positives
* 159 → False Negatives
* Model is making more mistakes on class 1 (minority class)

#**12. Prediction Function**

In [152]:
def predict_sentiment(tweet):
    tweet = clean_text(tweet)
    vec = vectorizer.transform([tweet])
    pred = model.predict(vec)[0]

    return "Positive" if pred == 0 else "Negative"

#**13. Test The Model**

In [153]:
print(predict_sentiment("This phone is amazing"))
print(predict_sentiment("Worst laptop ever"))
print(predict_sentiment("Absolutely love this laptop"))

Positive
Negative
Positive


**Interpretations**

The pipeline includes data cleaning, duplicate removal, text preprocessing, TF-IDF vectorization, and classification using Multinomial Naive Bayes. The model achieved 86% accuracy, with strong performance on the majority class and moderate performance on the minority class due to class imbalance.